In [7]:
import numpy as np
from pixell import enmap, curvedsky, utils

# ------------------------------------------------------------------
# 1. Paramètres généraux de la carte
# ------------------------------------------------------------------
size_deg = 10.0          # 10° x 10°
res_arcmin = 0.5         # résolution angulaire
res_rad = res_arcmin * utils.arcmin

npix = int(round(size_deg * 60.0 / res_arcmin))  # ≈ 1200

# Géométrie pixell (projection CAR, centrée sur RA=0°, Dec=0°)
shape, wcs = enmap.geometry(
    pos=(0.0 * utils.degree, 0.0 * utils.degree),
    shape=(npix, npix),
    res=res_rad,
    proj="car",
)
print("shape =", shape, "npix =", npix, "res_arcmin =", res_arcmin)

# ------------------------------------------------------------------
# 2. Chargement de la courbe de bruit SO baseline tSZ
# ------------------------------------------------------------------
so_noise_file = (
    "/rds/rds-clecat/pipeline_alina_full/alina_paper/"
    "sbi_wst_5000/SO_LAT_Nell_T_atmv1_baseline_fsky0p4_ILC_tSZ.txt"
)

# col 0 = ell, col 1 = N_ell^{yy} pour Deproj-0 (standard ILC tSZ)
ell_tab, Nell_tab = np.loadtxt(
    so_noise_file,
    unpack=True,
    usecols=(0, 1)   # on force à ne lire que les colonnes 0 et 1
)

# ell_tab : multipôles (ℓ)
# Nell_tab : N_ell^{yy} en unités de y^2, sans facteur ℓ(ℓ+1)/2π

# ------------------------------------------------------------------
# 3. Construction de N_ell sur tous les ℓ + fenêtre tanh 80<ℓ<7950
# ------------------------------------------------------------------
ell_max = int(ell_tab.max())
ells = np.arange(ell_max + 1)

# Interpolation linéaire de N_ell pour avoir une valeur à chaque ℓ entier
Nell = np.interp(ells, ell_tab, Nell_tab)

# Fenêtre tanh comme dans Alina (coupure douce en ℓ)
lmin, lmax = 80, 7950
delta = 10.0  # largeur de transition

def tanh_window(ell, lmin, lmax, delta):
    x_low = (ell - (lmin + 0.5 * delta)) / (0.5 * delta)
    f_low = 0.5 * (1.0 + np.tanh(x_low))
    x_high = ((lmax - 0.5 * delta) - ell) / (0.5 * delta)
    f_high = 0.5 * (1.0 + np.tanh(x_high))
    return f_low * f_high

window = tanh_window(ells, lmin, lmax, delta)
Cl_noise = Nell * window

# On met ℓ=0,1 à zéro (pas de sens pour un patch plat)
Cl_noise[:2] = 0.0

# ------------------------------------------------------------------
# 4. Préparation pour curvedsky.rand_map
# ------------------------------------------------------------------
ps = np.zeros((1, 1, Cl_noise.size))
ps[0, 0] = Cl_noise

# ------------------------------------------------------------------
# 5. Génération d'une carte de bruit tSZ (SO baseline)
# ------------------------------------------------------------------
noise_map = curvedsky.rand_map(
    shape, wcs, ps,
    lmax=ell_max,
    seed=None,     # fixe un int si tu veux une réalisation reproductible
)[0]

print("Carte de bruit générée, min/max =", noise_map.min(), noise_map.max())

# ------------------------------------------------------------------
# 6. Sauvegarde
# ------------------------------------------------------------------
out_file = "SO_baseline_ILC_tSZ_noise_10x10deg_0p5arcmin.fits"
enmap.write_map(out_file, noise_map)
print("Carte écrite dans", out_file)


shape = (1200, 1200) npix = 1200 res_arcmin = 0.5
Carte de bruit générée, min/max = -1.3975892190559746e-05 1.1962401499258158e-05
Carte écrite dans SO_baseline_ILC_tSZ_noise_10x10deg_0p5arcmin.fits


In [6]:
import numpy as np
from pixell import enmap, curvedsky, utils

# ------------------------------------------------------------------
# 1. Paramètres généraux de la carte
# ------------------------------------------------------------------
size_deg = 10.0          # 10° x 10°
res_arcmin = 0.5         # résolution angulaire
res_rad = res_arcmin * utils.arcmin

npix = int(round(size_deg * 60.0 / res_arcmin))  # ≈ 1200

# Géométrie pixell (projection CAR, centrée sur RA=0°, Dec=0°)
shape, wcs = enmap.geometry(
    pos=(0.0 * utils.degree, 0.0 * utils.degree),
    shape=(npix, npix),
    res=res_rad,
    proj="car",
)
print("shape =", shape, "npix =", npix, "res_arcmin =", res_arcmin)

# ------------------------------------------------------------------
# 2. Chargement de la courbe de bruit SO baseline tSZ
# ------------------------------------------------------------------
so_noise_file = (
    "/rds/rds-clecat/pipeline_alina_full/alina_paper/"
    "sbi_wst_5000/SO_LAT_Nell_T_atmv1_baseline_fsky0p4_ILC_tSZ.txt"
)

# col 0 = ell, col 1 = N_ell^{yy} pour Deproj-0 (standard ILC tSZ)
ell_tab, Nell_tab = np.loadtxt(
    so_noise_file,
    unpack=True,
    usecols=(0, 1)   # on force à ne lire que les colonnes 0 et 1
)

# ell_tab : multipôles (ℓ)
# Nell_tab : N_ell^{yy} en unités de y^2, sans facteur ℓ(ℓ+1)/2π

# ------------------------------------------------------------------
# 3. Construction de N_ell sur tous les ℓ + fenêtre tanh 80<ℓ<7950
# ------------------------------------------------------------------
ell_max = int(ell_tab.max())
ells = np.arange(ell_max + 1)

# Interpolation linéaire de N_ell pour avoir une valeur à chaque ℓ entier
Nell = np.interp(ells, ell_tab, Nell_tab)

# Fenêtre tanh comme dans Alina (coupure douce en ℓ)
lmin, lmax = 80, 7950
delta = 10.0  # largeur de transition

def tanh_window(ell, lmin, lmax, delta):
    x_low = (ell - (lmin + 0.5 * delta)) / (0.5 * delta)
    f_low = 0.5 * (1.0 + np.tanh(x_low))
    x_high = ((lmax - 0.5 * delta) - ell) / (0.5 * delta)
    f_high = 0.5 * (1.0 + np.tanh(x_high))
    return f_low * f_high

window = tanh_window(ells, lmin, lmax, delta)
Cl_noise = Nell * window

# On met ℓ=0,1 à zéro (pas de sens pour un patch plat)
Cl_noise[:2] = 0.0

# ------------------------------------------------------------------
# 4. Préparation pour curvedsky.rand_map
# ------------------------------------------------------------------
ps = np.zeros((1, 1, Cl_noise.size))
ps[0, 0] = Cl_noise

# ------------------------------------------------------------------
# 5. Génération d'une carte de bruit tSZ (SO baseline)
# ------------------------------------------------------------------
# On génère d'abord la carte complète (ncomp, ny, nx)
noise_map_full = curvedsky.rand_map(
    shape, wcs, ps,
    lmax=ell_max,
    seed=None,     # fixe un int si tu veux une réalisation reproductible
)

print("noise_map_full.shape =", noise_map_full.shape)

# On extrait la seule composante scalaire
noise_2d = noise_map_full[0]      # devrait être (ny, nx)
print("noise_2d.shape =", noise_2d.shape)

# Par sécurité, si jamais c'est aplati (1D), on reshape
if noise_2d.ndim == 1:
    print("ATTENTION : noise_2d est 1D, reshape en (npix, npix)")
    noise_2d = noise_2d.reshape(npix, npix)

# On force en enmap avec WCS
noise_enmap = enmap.enmap(noise_2d, wcs)   # <<< clé : enmap 2D avec WCS
print("noise_enmap.shape =", noise_enmap.shape, "ndim =", noise_enmap.ndim)

print("Carte de bruit générée, min/max =", noise_enmap.min(), noise_enmap.max())

# ------------------------------------------------------------------
# 6. Sauvegarde (enmap 2D correcte)
# ------------------------------------------------------------------
out_file = "SO_baseline_ILC_tSZ_noise_10x10deg_0p5arcmin.fits"
enmap.write_map(out_file, noise_enmap)     # <<< on écrit l'enmap, pas le tableau brut
print("Carte écrite dans", out_file)


shape = (1200, 1200) npix = 1200 res_arcmin = 0.5
noise_map_full.shape = (1200, 1200)
noise_2d.shape = (1200,)
ATTENTION : noise_2d est 1D, reshape en (npix, npix)


ValueError: cannot reshape array of size 1200 into shape (1200,1200)

In [9]:
import numpy as np
from pixell import enmap, curvedsky, utils

# ------------------------------------------------------------------
# 1. Paramètres généraux de la carte
# ------------------------------------------------------------------
size_deg = 10.0          # 10° x 10°
res_arcmin = 0.5         # résolution angulaire
res_rad = res_arcmin * utils.arcmin

npix = int(round(size_deg * 60.0 / res_arcmin))  # ≈ 1200

# Géométrie pixell (projection CAR, centrée sur RA=0°, Dec=0°)
shape, wcs = enmap.geometry(
    pos=(0.0 * utils.degree, 0.0 * utils.degree),
    shape=(npix, npix),
    res=res_rad,
    proj="car",
)
print("shape =", shape, "npix =", npix, "res_arcmin =", res_arcmin)

# ------------------------------------------------------------------
# 2. Chargement de la courbe de bruit SO baseline tSZ
# ------------------------------------------------------------------
so_noise_file = (
    "/rds/rds-clecat/pipeline_alina_full/alina_paper/"
    "sbi_wst_5000/SO_LAT_Nell_T_atmv1_baseline_fsky0p4_ILC_tSZ.txt"
)

# col 0 = ell, col 1 = N_ell^{yy} pour Deproj-0 (standard ILC tSZ)
ell_tab, Nell_tab = np.loadtxt(
    so_noise_file,
    unpack=True,
    usecols=(0, 1)
)

# ------------------------------------------------------------------
# 3. Construction de N_ell sur tous les ℓ + fenêtre tanh 80<ℓ<7950
# ------------------------------------------------------------------
ell_max = int(ell_tab.max())
ells    = np.arange(ell_max + 1)

Nell = np.interp(ells, ell_tab, Nell_tab)

lmin, lmax = 80, 7950
delta      = 10.0

def tanh_window(ell, lmin, lmax, delta):
    x_low  = (ell - (lmin + 0.5 * delta)) / (0.5 * delta)
    f_low  = 0.5 * (1.0 + np.tanh(x_low))
    x_high = ((lmax - 0.5 * delta) - ell) / (0.5 * delta)
    f_high = 0.5 * (1.0 + np.tanh(x_high))
    return f_low * f_high

window   = tanh_window(ells, lmin, lmax, delta)
Cl_noise = Nell * window
Cl_noise[:2] = 0.0

# ------------------------------------------------------------------
# 4. Préparation pour curvedsky.rand_map
# ------------------------------------------------------------------
ps       = np.zeros((1, 1, Cl_noise.size))
ps[0, 0] = Cl_noise

# ------------------------------------------------------------------
# 5. Génération d'une carte de bruit tSZ (SO baseline)
# ------------------------------------------------------------------
# ATTENTION : on ne met plus [0] ici !
noise_map = curvedsky.rand_map(
    shape, wcs, ps,
    lmax=ell_max,
    seed=None,
)

print("noise_map.shape =", noise_map.shape, "ndim =", noise_map.ndim)
print("Carte de bruit générée, min/max =", noise_map.min(), noise_map.max())

# ------------------------------------------------------------------
# 6. Sauvegarde
# ------------------------------------------------------------------
out_file = "SO_baseline_ILC_tSZ_noise_10x10deg_0p5arcmin.fits"
enmap.write_map(out_file, noise_map)
print("Carte écrite dans", out_file)


shape = (1200, 1200) npix = 1200 res_arcmin = 0.5
noise_map.shape = (1200, 1200) ndim = 2
Carte de bruit générée, min/max = -2.0137006258219302e-05 2.1039470529970072e-05
Carte écrite dans SO_baseline_ILC_tSZ_noise_10x10deg_0p5arcmin.fits


In [10]:
import numpy as np
from pixell import enmap, curvedsky, utils

# ------------------------------------------------------------------
# 1. Paramètres généraux de la carte
# ------------------------------------------------------------------
size_deg   = 10.0          # 10° x 10°
res_arcmin = 0.5           # résolution angulaire
res_rad    = res_arcmin * utils.arcmin

npix = int(round(size_deg * 60.0 / res_arcmin))  # ≈ 1200

# Géométrie pixell (projection CAR, centrée sur RA=0°, Dec=0°)
shape, wcs = enmap.geometry(
    pos=(0.0 * utils.degree, 0.0 * utils.degree),
    shape=(npix, npix),
    res=res_rad,
    proj="car",
)
print("shape =", shape, "npix =", npix, "res_arcmin =", res_arcmin)

# ------------------------------------------------------------------
# 2. Chargement de la courbe de bruit SO baseline tSZ
# ------------------------------------------------------------------
so_noise_file = (
    "/rds/rds-clecat/pipeline_alina_full/alina_paper/"
    "sbi_wst_5000/SO_LAT_Nell_T_atmv1_baseline_fsky0p4_ILC_tSZ.txt"
)

ell_tab, Nell_tab = np.loadtxt(
    so_noise_file,
    unpack=True,
    usecols=(0, 1)   # ℓ et Nℓ^{yy} standard ILC tSZ
)

# ------------------------------------------------------------------
# 3. Construction de N_ell + fenêtre tanh 80<ℓ<7950
# ------------------------------------------------------------------
ell_max = int(ell_tab.max())
ells    = np.arange(ell_max + 1)

Nell = np.interp(ells, ell_tab, Nell_tab)

lmin, lmax = 80, 7950
delta      = 10.0

def tanh_window(ell, lmin, lmax, delta):
    x_low  = (ell - (lmin + 0.5 * delta)) / (0.5 * delta)
    f_low  = 0.5 * (1.0 + np.tanh(x_low))
    x_high = ((lmax - 0.5 * delta) - ell) / (0.5 * delta)
    f_high = 0.5 * (1.0 + np.tanh(x_high))
    return f_low * f_high

window   = tanh_window(ells, lmin, lmax, delta)
Cl_noise = Nell * window
Cl_noise[:2] = 0.0

ps       = np.zeros((1, 1, Cl_noise.size))
ps[0, 0] = Cl_noise

# ------------------------------------------------------------------
# 4. Génération de la carte de bruit tSZ (SO baseline)
# ------------------------------------------------------------------
noise_map = curvedsky.rand_map(
    shape, wcs, ps,
    lmax=ell_max,
    seed=None,
)

print("noise_map.shape =", noise_map.shape, "ndim =", noise_map.ndim)

# Par sécurité, on force en enmap (au cas où la version de pixell renverrait un ndmap/ndarray brut)
noise_map = enmap.enmap(noise_map, wcs)
print("type(noise_map) =", type(noise_map))

print("Carte de bruit générée, min/max =", noise_map.min(), noise_map.max())

# ------------------------------------------------------------------
# 5. Sauvegarde sous un NOUVEAU NOM
# ------------------------------------------------------------------
out_file = (
    "/rds/rds-clecat/pipeline_alina_full/alina_paper/sbi_wst_5000/"
    "SO_baseline_ILC_tSZ_noise_10x10deg_0p5arcmin_v2.fits"
)
enmap.write_map(out_file, noise_map)
print("Carte écrite dans", out_file)

# ------------------------------------------------------------------
# 6. Test immédiat de relecture
# ------------------------------------------------------------------
test_map = enmap.read_map(out_file)
print("Relecture OK.")
print("test_map.shape =", test_map.shape, "ndim =", test_map.ndim, "type =", type(test_map))


shape = (1200, 1200) npix = 1200 res_arcmin = 0.5
noise_map.shape = (1200, 1200) ndim = 2
type(noise_map) = <class 'pixell.enmap.ndmap'>
Carte de bruit générée, min/max = -1.919030312516039e-05 2.0785629785560198e-05
Carte écrite dans /rds/rds-clecat/pipeline_alina_full/alina_paper/sbi_wst_5000/SO_baseline_ILC_tSZ_noise_10x10deg_0p5arcmin_v2.fits
Relecture OK.
test_map.shape = (1200, 1200) ndim = 2 type = <class 'pixell.enmap.ndmap'>


In [11]:
from astropy.io import fits
from pixell import enmap

noise = enmap.read_map("SO_baseline_ILC_tSZ_noise_10x10deg_0p5arcmin_v2.fits")
signal = enmap.read_map("y_map_10.0deg_0.500arcmin_235.fits")

print(noise.shape, signal.shape)
print(noise.wcs)
print(signal.wcs)


FileNotFoundError: [Errno 2] No such file or directory: 'y_map_10.0deg_0.500arcmin_235.fits'

In [13]:
from astropy.io import fits
from pixell import enmap

# --- Chemins absolus complets ---
noise = enmap.read_map("/rds/rds-clecat/pipeline_alina_full/alina_paper/sbi_wst_5000/SO_baseline_ILC_tSZ_noise_10x10deg_0p5arcmin_v2.fits")

signal = enmap.read_map("/rds/rds-clecat/pipeline_alina_full/alina_paper/pipeline_outputs/paint_cov_big_sim/logA=2.96_Oc0h2=0.105217/y_map_10.0deg_0.500arcmin_235.fits")

# --- Vérifications WCS + dimensions ---
print(noise.shape, signal.shape)
print(noise.wcs)
print(signal.wcs)


(1200, 1200) (1200, 1200)
car:{cdelt:[-0.008333,0.008333],crval:[0,0],crpix:[601.00,601.00]}
car:{cdelt:[-0.008333,0.008333],crval:[0,0],crpix:[601.00,601.00]}


In [12]:
#Adding the two maps

In [14]:
from pixell import enmap

# --- Chemins absolus fournis par vous ---
noise_path  = "/rds/rds-clecat/pipeline_alina_full/alina_paper/sbi_wst_5000/SO_baseline_ILC_tSZ_noise_10x10deg_0p5arcmin_v2.fits"

signal_path = "/rds/rds-clecat/pipeline_alina_full/alina_paper/pipeline_outputs/paint_cov_big_sim/logA=2.96_Oc0h2=0.105217/y_map_10.0deg_0.500arcmin_235.fits"

# --- Lecture des cartes ---
noise  = enmap.read_map(noise_path)
signal = enmap.read_map(signal_path)

# --- Vérification / reprojection si nécessaire ---
if (noise.shape != signal.shape) or (noise.wcs != signal.wcs):
    noise = noise.project(signal.shape, signal.wcs)

# --- Addition bruit + signal ---
combined = signal + noise

# --- Chemin absolu de sauvegarde ---
output_path = "/rds/rds-clecat/pipeline_alina_full/alina_paper/pipeline_outputs/paint_cov_big_sim/logA=2.96_Oc0h2=0.105217/y_map_with_noise_235.fits"

enmap.write_map(output_path, combined)

print(f"Carte combinée sauvegardée dans : {output_path}")


Carte combinée sauvegardée dans : /rds/rds-clecat/pipeline_alina_full/alina_paper/pipeline_outputs/paint_cov_big_sim/logA=2.96_Oc0h2=0.105217/y_map_with_noise_235.fits


In [16]:
decombined = combined - noise
output_path = "/rds/rds-clecat/pipeline_alina_full/alina_paper/pipeline_outputs/paint_cov_big_sim/logA=2.96_Oc0h2=0.105217/test_recovery.fits"

enmap.write_map(output_path, decombined)

print(f"Carte combinée sauvegardée dans : {output_path}")

Carte combinée sauvegardée dans : /rds/rds-clecat/pipeline_alina_full/alina_paper/pipeline_outputs/paint_cov_big_sim/logA=2.96_Oc0h2=0.105217/test_recovery.fits
